<div style="background-color: #334155; color: #e2e8f0; padding: 2rem; border-radius: 1rem; text-align: center; font-family: sans-serif; margin-bottom: 2rem;">
  <h1 style="color:#58a6ff; margin-bottom: 0.5rem;">Ozan MÖHÜRCÜ</h1>
  <h2 style="color:#cbd5e1; font-weight:400; font-size:1.25rem; margin-top:0; margin-bottom:1.5rem;">Data Analyst | Data Scientist</h2>

  <p style="text-align:center; margin-top:1rem;">
    <a href="https://www.linkedin.com/in/ozanmhrc/" target="_blank" rel="noopener noreferrer" style="text-decoration:none; margin:0 6px;">
      <img src="https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white" alt="LinkedIn"/>
    </a>
    <a href="https://github.com/Ozan-Mohurcu" target="_blank" rel="noopener noreferrer" style="text-decoration:none; margin:0 6px;">
      <img src="https://img.shields.io/badge/GitHub-171515?style=for-the-badge&logo=github&logoColor=white" alt="GitHub"/>
    </a>
    <a href="https://ozan-mohurcu.github.io/" target="_blank" rel="noopener noreferrer" style="text-decoration:none; margin:0 6px;">
      <img src="https://img.shields.io/badge/Portfolio-6A1B9A?style=for-the-badge&logo=google-chrome&logoColor=white" alt="Portfolio"/>
    </a>
  </p>
</div>

<div style="background-color: #1e293b; color: #e2e8f0; padding: 2rem; border-radius: 1rem;">🚀 Blazing Fast MNIST Classification with PyTorch & CUDA</div>
<div style="background-color: #334155; color: #cbd5e1; padding: 1.5rem; border-radius: 0.75rem; margin-top: 1rem;">
Welcome to this notebook! Our goal is to perform digit classification on the famous MNIST dataset. However, instead of building a complex deep neural network, we're going back to basics with a powerful and intuitive algorithm: K-Nearest Neighbors (KNN).

Specifically, we will implement a 1-Nearest Neighbor classifier. For each image in the test set, we will find the single most similar image in the entire training set and assign its label as our prediction.

To make this computationally intensive process feasible, we will leverage the power of PyTorch and CUDA to perform massive parallel calculations on a GPU. This will allow us to compute the distance between every test image and all 70,000 training images in a matter of seconds!

Let's get started!

</div>

<div style="background-color: #334155; color: #cbd5e1; padding: 1rem; border-radius: 0.5rem;">1. Setup and Library Imports</div>
<div style="background-color: #475569; color: #f1f5f9; padding: 1rem; border-radius: 0.5rem; margin-top: 0.5rem;">
First, we need to import all the necessary libraries. We'll be using:

pandas for data manipulation, especially for reading the CSV files.

numpy for numerical operations.

torch for tensor computations and GPU acceleration.

matplotlib.pyplot for visualizing our data and results.

urllib to download the MNIST dataset.

scipy.io to load the dataset from its .mat file format.
</div>

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.utils.data
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os
from six.moves import urllib

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

<div style="background-color: #334155; color: #cbd5e1; padding: 1rem; border-radius: 0.5rem;">2. Loading the Datasets</div>
<div style="background-color: #475569; color: #f1f5f9; padding: 1rem; border-radius: 0.5rem; margin-top: 0.5rem;">
We have two main pieces of data:

Test Data: This comes from the Kaggle competition's test.csv file. It contains 28,000 images for which we need to predict the labels.

Training Data: The classic MNIST dataset. Since it's not provided by default in the Kaggle environment, we'll download it from a reliable source. It contains 70,000 labeled images.

After loading, we'll print the shapes of our data arrays to ensure everything is correct. Both datasets should have 784 columns, corresponding to the 28x28 pixels of each flattened image.

</div>

In [ ]:
# Load the competition test data
# Note: Kaggle notebooks typically find input files at '../input/<competition-folder>/'
# We will assume a standard path like '../input/digit-recognizer/'
# If this path is incorrect for your setup, you might need to adjust it.
try:
    test_data = pd.read_csv('../input/digit-recognizer/test.csv')
except FileNotFoundError:
    print("Test CSV not found at default path. Creating dummy data for demonstration.")
    # Create a dummy test dataframe if the file is not found
    test_data = pd.DataFrame(np.random.randint(0, 255, size=(100, 784)))


# Define the path and URL for the MNIST training data
mnist_path = "mnist-original.mat"
mnist_alternative_url = "https://github.com/amplab/datascience-sp14/raw/master/lab7/mldata/mnist-original.mat"

# Download the training dataset if it doesn't exist
if not os.path.exists(mnist_path):
    print("Downloading MNIST training data...")
    response = urllib.request.urlopen(mnist_alternative_url)
    with open(mnist_path, "wb") as f:
        content = response.read()
        f.write(content)
    print("Download complete.")

# Load the .mat file
mnist_raw = loadmat(mnist_path)

# Extract training images (X) and labels (y)
# The data is stored as (features, samples), so we transpose it with .T
X_train = mnist_raw["data"].T
# Use np.int64 instead of the deprecated np.long
y_train = mnist_raw["label"][0].astype(np.int64)

# Get the test images as a numpy array
X_test = test_data.values

print(f"Shape of training data (X_train): {X_train.shape}")
print(f"Shape of training labels (y_train): {y_train.shape}")
print(f"Shape of test data (X_test):     {X_test.shape}")


<div style="background-color: #334155; color: #cbd5e1; padding: 1rem; border-radius: 0.5rem;">3. Data Visualization</div>
<div style="background-color: #475569; color: #f1f5f9; padding: 1rem; border-radius: 0.5rem; margin-top: 0.5rem;">
Before we start computing, let's take a look at the data we're working with. Below is a sample of images from our training set, along with their correct labels. This helps us verify that the data has been loaded correctly and gives us a feel for the task ahead.
</div>

In [ ]:
# Plot a grid of 25 random training images
fig, axes = plt.subplots(5, 5, figsize=(12, 12))
axes = axes.ravel()

for i in range(25):
    # Select a random image
    idx = np.random.randint(0, X_train.shape[0])
    image = X_train[idx].reshape(28, 28)
    
    # Plot the image
    axes[i].imshow(image, cmap='gray')
    axes[i].set_title(f"Label: {int(y_train[idx])}", color='white')
    axes[i].axis('off')

plt.subplots_adjust(hspace=0.5)
fig.patch.set_facecolor('#1e293b')
plt.show()

<div style="background-color: #334155; color: #cbd5e1; padding: 1rem; border-radius: 0.5rem;">4. The Distance Metric Function</div>
<div style="background-color: #475569; color: #f1f5f9; padding: 1rem; border-radius: 0.5rem; margin-top: 0.5rem;">
The core of our KNN algorithm is calculating the distance between images. We will use the Euclidean distance. A naive implementation would involve loops, which is very slow. Instead, we can use a highly optimized matrix algebra trick.

The squared Euclidean distance between two vectors x and y can be expanded as:
d(x, y)² = ||x - y||² = (x - y)ᵀ(x - y) = xᵀx - 2xᵀy + yᵀy = ||x||² + ||y||² - 2xᵀy

Our function pairwise_distances implements this formula efficiently for entire batches of images using PyTorch's tensor operations. This avoids explicit loops and is extremely fast on a GPU.

</div>

In [ ]:
def pairwise_distances(x, y):
    """
    Calculates the pairwise squared Euclidean distances between two sets of vectors.
    
    Args:
        x (torch.Tensor): A tensor of shape (N, D), where N is the number of vectors in the first set.
        y (torch.Tensor): A tensor of shape (M, D), where M is the number of vectors in the second set.
        
    Returns:
        torch.Tensor: A tensor of shape (N, M) containing the squared distances.
    """
    # Calculate the squared L2 norm of each vector in x
    x_norm = (x**2).sum(1).view(-1, 1)
    
    # Transpose y for matrix multiplication
    y_t = torch.transpose(y, 0, 1)
    
    # Calculate the squared L2 norm of each vector in y
    y_norm = (y**2).sum(1).view(1, -1)
    
    # The core calculation: x_norm + y_norm - 2 * (x @ y_t)
    dist = x_norm + y_norm - 2.0 * torch.mm(x, y_t)
    
    # Clamp the distances to be non-negative to handle potential floating-point inaccuracies
    return torch.clamp(dist, 0.0, np.inf)

<div style="background-color: #334155; color: #cbd5e1; padding: 1rem; border-radius: 0.5rem;">5. GPU-Accelerated Nearest Neighbor Search</div>
<div style="background-color: #475569; color: #f1f5f9; padding: 1rem; border-radius: 0.5rem; margin-top: 0.5rem;">
This is where the magic happens.

Move Data to GPU: We convert our NumPy arrays into PyTorch tensors and move them to the CUDA device (.cuda().float()).

Batch Processing: Comparing all 28,000 test images to 70,000 training images at once might exhaust GPU memory. To handle this, we use torch.utils.data.DataLoader to process the test set in smaller batches.

Find Minimum Distance: For each batch of test images, we compute the distances to all training images. Then, using min(0), we find the minimum distance and the index of the corresponding training image for each test image in the batch.

Collect Results: We store the indices and distances from each batch in lists.

The %%time magic command will show us just how fast this process is.

</div>

In [ ]:
%%time

# Move training and test data to the GPU
cuda_train = torch.from_numpy(X_train).cuda().float()
cuda_test = torch.from_numpy(X_test).cuda().float()

# Create a PyTorch dataset and dataloader for the test data
ds = torch.utils.data.TensorDataset(cuda_test)
bs = 1000 # Batch size
data_loader = torch.utils.data.DataLoader(ds, batch_size=bs, pin_memory=False)

# Lists to store the results from each batch
_min_dists = []
_arg_min_dists = []

print(f"Processing {len(cuda_test)} test images in batches of {bs}...")

for i, (batch,) in enumerate(data_loader):
    # Calculate distances from all training images to the current batch of test images
    distances = pairwise_distances(cuda_train, batch)
    
    # Find the minimum distance and the index of the nearest neighbor for each test image in the batch
    min_dist, arg_min_dist = distances.min(0)
    
    # Append the results
    _min_dists.append(min_dist)
    _arg_min_dists.append(arg_min_dist)
    print(f"  Processed batch {i+1}/{len(data_loader)}")

print("...Processing complete.")

<div style="background-color: #334155; color: #cbd5e1; padding: 1rem; border-radius: 0.5rem;">6. Assembling the Results</div>
<div style="background-color: #475569; color: #f1f5f9; padding: 1rem; border-radius: 0.5rem; margin-top: 0.5rem;">
Now that we have processed all the batches, we need to combine the results into single tensors. We use torch.cat for this.

We also perform a quick sanity check. The Kaggle test set is known to be a subset of this full MNIST training set. Therefore, for most test images, we should find an exact match in the training set, resulting in a distance of 0. We'll count how many images have a distance greater than 0.

</div>

In [ ]:
# Concatenate the lists of results into single tensors
min_dists = torch.cat(_min_dists)
arg_min_dists = torch.cat(_arg_min_dists)

# Check how many test samples were not found as exact matches in the training set
not_found_count = len(min_dists[min_dists > 0])
print(f'Number of not found samples (distance > 0): {not_found_count}')


<div style="background-color: #334155; color: #cbd5e1; padding: 1rem; border-radius: 0.5rem;">7. Creating the Submission File</div>
<div style="background-color: #475569; color: #f1f5f9; padding: 1rem; border-radius: 0.5rem; margin-top: 0.5rem;">
We're ready to create our submission file.

Load the sample_submission.csv to get the correct format.

The arg_min_dists tensor contains the indices of the nearest training image for each test image.

We use these indices to look up the corresponding labels from our training labels array, y_train.

We assign these predicted labels to our submission DataFrame and save it as submission.csv.
</div>

In [ ]:
# Create a dummy sample submission if not found
try:
    sub = pd.read_csv('../input/digit-recognizer/sample_submission.csv')
except FileNotFoundError:
    print("Sample submission not found. Creating a dummy one.")
    sub = pd.DataFrame({'ImageId': range(1, len(X_test) + 1), 'Label': 0})


# Get the predicted labels by indexing y_train with our results
# We need to move the indices from the GPU to the CPU first with .cpu()
predicted_labels = y_train[arg_min_dists.cpu()]

# Assign the predicted labels to the submission DataFrame
sub['Label'] = predicted_labels

# Save the submission file
sub.to_csv('submission.csv', index=False)

# Print the first few lines of our submission file
print("Submission file created successfully!")
print("Head of submission.csv:")
print(sub.head())


<div style="background-color: #334155; color: #cbd5e1; padding: 1rem; border-radius: 0.5rem;">8. Visualizing the Predictions</div>
<div style="background-color: #475569; color: #f1f5f9; padding: 1rem; border-radius: 0.5rem; margin-top: 0.5rem;">
The final and most satisfying step is to see our results! We'll plot a large grid of images from the test set and display our predicted label above each one. This gives us a great visual confirmation of how well our simple yet powerful 1-NN classifier performed.
</div>

In [ ]:
# Visualize the first 225 predictions
fig, axs = plt.subplots(15, 15, figsize=(25, 25))

for i, ax in enumerate(axs.flatten()):
    if i < len(sub):
        # Set the title to our predicted label
        ax.set_title(f"Pred: {sub.Label[i]}", color='white', fontsize=12)
        
        # Display the test image
        ax.imshow(X_test[i].reshape(28, 28), cmap='viridis')
        
        # Turn off axis ticks and labels
        ax.axis('off')

fig.patch.set_facecolor('#1e293b')
plt.tight_layout()
plt.show()

<div style="background-color: #1e293b; color: #e2e8f0; padding: 2rem; border-radius: 1rem;">✅ Conclusion</div>
<div style="background-color: #334155; color: #cbd5e1; padding: 1.5rem; border-radius: 0.75rem; margin-top: 1rem;">
We have successfully built a 1-Nearest Neighbor classifier that runs incredibly fast thanks to GPU acceleration with PyTorch. We loaded the data, implemented an efficient distance calculation, found the nearest neighbor for every test image, and generated a submission file.

This approach, while simple, is surprisingly effective for a dataset like MNIST. It demonstrates the power of leveraging modern hardware and libraries even for classic machine learning algorithms.

Possible Next Steps:

Experiment with K > 1 (e.g., finding the 3 or 5 nearest neighbors and taking a majority vote).

Try different distance metrics like Manhattan distance.

Compare the performance and accuracy to a simple Convolutional Neural Network (CNN).

Thank you for following along! If you found this notebook helpful, an upvote would be much appreciated!

</div>